In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import copy
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, normalize
import scipy.cluster.hierarchy as shc
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import GridSearchCV
from sklearn import metrics
from tqdm import tqdm
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tensorboardX import SummaryWriter
import os
import time
from torcheval.metrics import R2Score
from sklearn.metrics import r2_score
import warnings
import pickle 
from scipy import stats


%matplotlib inline
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore")

In [2]:
def swap_columns(df, col1, col2):
    col_list = list(df.columns)
    x, y = col_list.index(col1), col_list.index(col2)
    col_list[y], col_list[x] = col_list[x], col_list[y]
    df = df[col_list]
    return df

# pipeline

In [3]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        if not torch.is_tensor(X) or not torch.is_tensor(y):
            X = torch.from_numpy(X)
            y = torch.from_numpy(y)
            
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


class MLP(nn.Module):
    '''
        Multilayer Perceptron for regression.
    '''
    def __init__(self, n_in, n_out):
        super().__init__()
#         self.layers = nn.Sequential(
#             nn.Linear(n_in, 64),
#             nn.ReLU(),
#             nn.Linear(64, 32),
#             nn.ReLU(),
#             nn.Linear(32, n_out)
#         )
        
#         self.layers = nn.Sequential(
#             nn.Linear(n_in, 32),
#             nn.ReLU(),
#             nn.Linear(32, 64),
#             nn.ReLU(),
#             nn.Linear(64, 64),
#             nn.ReLU(),
#             nn.Linear(64, 32),
#             nn.ReLU(),
#             nn.Linear(32, 16),
#             nn.ReLU(),
#             nn.Linear(16, n_out)
#         )

        self.layers = nn.Sequential(
            nn.Linear(n_in, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, n_out)
        )

#         self.layers = nn.Sequential(
#             nn.Linear(n_in, 32),
#             nn.ReLU(),
#             nn.Linear(32, 32),
#             nn.ReLU(),
#             nn.Linear(32, 32),
#             nn.ReLU(),
#             nn.Linear(32, n_out)
#         )


    def forward(self, x):
        '''
          Forward pass
        '''
        return self.layers(x)

In [4]:
def run_deep(X, y, idx, test_size, 
             train_batch_size, val_batch_size,
             learning_rate,
             tensorboard_path,
             print_iteration=1,
             n_epochs=100,
             tensorboard=True, 
             do_print=True):

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=test_size)

    # Prepare dataset
    train_dataset = Dataset(X_train, y_train)
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True)

    val_dataset = Dataset(X_val, y_val)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=val_batch_size, shuffle=True)

    # Initialize the MLP
    mlp = MLP(n_in=X_train.shape[1], n_out=1)

    # Define the loss function and optimizer
    loss_function = nn.MSELoss(reduction='sum')
    optimizer = torch.optim.Adam(mlp.parameters(), lr=learning_rate)    
    
    if tensorboard:
        writer = SummaryWriter(tensorboard_path)
    
    # Run the training loop
    total_iteration = 0
    best_val_results = dict(loss = np.inf, corr = -np.inf, r2 = -np.inf)
    for epoch in range(n_epochs): 
        outputs_train_arr = np.array([])
        targets_train_arr = np.array([])

        current_loss = 0.0
        n_train_data = 0
        for inputs, targets in train_loader:
            total_iteration += 1
            inputs, targets = inputs.float(), targets.float()
            targets = targets.reshape((targets.shape[0], 1))
            n_train_data += inputs.shape[0]
            
            optimizer.zero_grad()
            
            outputs = mlp(inputs)
            loss = loss_function(outputs, targets)
            loss.backward()
            optimizer.step()
            current_loss += loss.item()
            
            outputs_detach = torch.detach(outputs)
            targets_detach = torch.detach(targets)
            outputs_train_arr = np.append(outputs_train_arr, torch.Tensor.numpy(outputs_detach).reshape(1,-1)[0])
            targets_train_arr = np.append(targets_train_arr, torch.Tensor.numpy(targets_detach).reshape(1,-1)[0])
                       
            if total_iteration % print_iteration == 0:
                loss_val = 0
                n_val_data = 0
                outputs_val_arr = np.array([])
                targets_val_arr = np.array([])

                for inputs_val, targets_val in val_loader:
                    inputs_val, targets_val = inputs_val.float(), targets_val.float()
                    targets_val = targets_val.reshape((targets_val.shape[0], 1))
                    n_val_data += inputs_val.shape[0]
                    outputs_val = mlp(inputs_val)
                    loss_val += loss_function(outputs_val, targets_val).item()
                    outputs_val_detach = torch.detach(outputs_val)
                    targets_val_detach = torch.detach(targets_val)
                    outputs_val_arr = np.append(outputs_val_arr, torch.Tensor.numpy(outputs_val_detach).reshape(1,-1)[0])
                    targets_val_arr = np.append(targets_val_arr, torch.Tensor.numpy(targets_val_detach).reshape(1,-1)[0])
                    
                corrcoef = np.corrcoef(targets_val_arr, outputs_val_arr)[0, 1]
                r_squared = r2_score(targets_val_arr, outputs_val_arr)
                train_loss = current_loss / n_train_data
                val_loss = loss_val / n_val_data
                
                if tensorboard:
                    writer.add_scalars(f'{str(idx)}_loss', {
                        'train': train_loss,
                        'validation': val_loss,
                    }, total_iteration)
                    writer.add_scalars(f'{str(idx)}_corr-R', {
                        'corr': corrcoef,
                        'r': r_squared,
                    }, total_iteration)
                
                if val_loss < best_val_results['loss']:
                    best_val_results['loss']
                best_val_results['loss'] = val_loss if val_loss < best_val_results['loss'] else best_val_results['loss']
                best_val_results['corr'] = corrcoef if corrcoef > best_val_results['corr'] else best_val_results['corr']
                best_val_results['r2'] = r_squared if r_squared > best_val_results['r2'] else best_val_results['r2']
                
                current_loss = 0.0
                n_train_data = 0
        
        corrcoef = np.corrcoef(targets_train_arr, outputs_train_arr)[0, 1]
        r_squared = r2_score(targets_train_arr, outputs_train_arr)
        
    print_str = (f"best of {str(idx).rjust(3)} => loss: {str(round(best_val_results['loss'], 2)).rjust(11)}, " + 
                 f"corr: {str(round(best_val_results['corr'], 2)).rjust(5)}, " + 
                 f"r2: {str(round(best_val_results['r2'], 2)).rjust(5)}")
    if do_print:
        print(print_str)
    
    if tensorboard:
        writer.add_text(str(idx), print_str)
        del writer
        
    return print_str, best_val_results

# run pipeline

In [6]:
# data

# df_to_run = pd.read_csv('data/deep_test_retest_imputed.csv')
df_to_run = pd.read_csv('../../data/deep_test_retest_imputed_measures_standardized.csv')
data_arr = df_to_run.iloc[:,2:].to_numpy()
X = data_arr[:, :12]
y_total = data_arr[:, 12:]

In [36]:
X_df.drop(columns='age').to_numpy()

array([[  0. ,   4. ,   0. , ...,  50. ,  80. , 126. ],
       [  0. ,   0.5,   0. , ...,  61. ,  80. , 135. ],
       [  1. ,   0. ,   0. , ...,  42. ,  40. ,  84. ],
       ...,
       [  1. ,   3. ,   0. , ...,  61. ,  72. , 129. ],
       [  0. ,   1. ,   0. , ...,  66. ,  73. , 134. ],
       [  0. ,   0. ,   0. , ...,  64. ,  66. , 126. ]])

In [18]:
array = np.arange(X.shape[1])
array

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])

In [26]:
for col, arr in zip(X_df.columns, X.T):
    print(col, type(arr))

age <class 'numpy.ndarray'>
gender <class 'numpy.ndarray'>
musicAllYears <class 'numpy.ndarray'>
musicFormal <class 'numpy.ndarray'>
cog_wm_omissions <class 'numpy.ndarray'>
cog_wm_rt <class 'numpy.ndarray'>
cog_flex_totalperf <class 'numpy.ndarray'>
cog_gonogo_errors <class 'numpy.ndarray'>
cog_gonogo_rt <class 'numpy.ndarray'>
wasiMatrixT <class 'numpy.ndarray'>


In [28]:
df_to_run

,subject_session,ageGroup,age,gender,musicAllYears,musicFormal,cog_wm_omissions,cog_wm_rt,cog_flex_totalperf,cog_gonogo_errors,cog_gonogo_rt,wasiMatrixT,wasiVocabT,wasiFSIQ2,BAT_all_dprime,BAT_fast_dprime,BAT_med_dprime,BAT_slow_dprime,BAT_all_hits,BAT_fast_hits,BAT_med_hits,BAT_slow_hits,BAT_all_fa,BAT_fast_fa,BAT_med_fa,BAT_slow_fa,DurDisc_threshold,Anisochrony_tones_threshold,Anisochrony_music_threshold,Unpaced_spontaneous_right_CV_iti,Unpaced_spontaneous_final_right_CV_iti,Unpaced_fast_CV_iti,Unpaced_slow_CV_iti,Unpaced_spontaneous_right_mean_iti,Unpaced_spontaneous_final_right_mean_iti,Unpaced_fast_mean_iti,Unpaced_slow_mean_iti,Paced_metro_450_mean_CV_iti,Paced_metro_450_vecDirPct,Paced_metro_450_mean_vector_direction,Paced_metro_450_vecLenLogit,Paced_metro_600_mean_CV_iti,Paced_metro_600_vecDirPct,Paced_metro_600_mean_vector_direction,Paced_metro_600_vecLenLogit,Paced_metro_750_mean_CV_iti,Paced_metro_750_vecDirPct,Paced_metro_750_mean_vector_direction,Paced_metro_750_vecLenLogit,Paced_music_badine_mean_CV_iti,Paced_music_badine_vecDirPct,Paced_music_badine_mean_vector_direction,Paced_music_badine_vecLenLogit,Paced_music_ross_mean_CV_iti,Paced_music_ross_vecDirPct,Paced_music_ross_mean_vector_direction,Paced_music_ross_vecLenLogit,Adaptive_adaptation_index_acceleration,Adaptive_adaptation_index_deceleration,Adaptive_iso_600_CV_iti,Adaptive_minus_30_dprime2,Adaptive_minus_75_dprime2,Adaptive_plus_30_dprime2,Adaptive_plus_75_dprime2,SyncCont_metro_450_mean_CV_iti,SyncCont_metro_600_mean_CV_iti,SyncCont_metro_750_mean_CV_iti,SyncCont_metro_450_mean_mean_iti,SyncCont_metro_600_mean_mean_iti,SyncCont_metro_750_mean_mean_iti
0,A03_PRE,Age_22_29,22,0,4.0,0.0,44.0,43.0,58.0,40.0,52.0,50.0,80.0,126.0,-1.117825,-1.159666,-1.162388,-0.384372,0.559051,0.621256,0.162754,0.742530,3.082329,2.706190,2.519201,2.368778,0.286102,-1.390923,-0.369176,0.600602,5.903224,0.467735,0.445230,0.480356,0.748724,0.649342,1.677309,1.215221,-0.777824,-0.541681,-1.594148,3.376893,0.464756,0.466648,-0.639599,1.294873,-0.315105,-0.315105,-0.181463,0.544694,-1.002731,-0.831893,-0.071738,0.656663,-0.788603,-0.585061,-0.127144,-0.346649,-1.564702,2.078664,-2.211761,-1.562522,-2.168695,-4.136652,2.259454,3.886848,2.088498,-1.697993,0.775256,0.443109
1,A04_PRE,Age_22_29,22,0,0.5,0.0,36.0,41.0,61.0,47.0,43.0,61.0,80.0,135.0,0.824157,0.826215,0.490315,0.958353,0.705587,0.621256,0.542514,0.742530,-0.650975,-0.657117,-0.463733,-0.422159,-0.601409,-0.487563,-0.779771,-0.133470,-0.555041,0.282094,-0.146414,-0.749148,-0.507287,-0.623180,0.028125,0.011330,0.901992,0.850409,0.335237,-0.570282,0.855084,0.812029,-0.094977,0.892137,1.251825,1.251825,0.205237,-0.019815,0.414078,0.385741,0.264570,-0.587560,1.261220,1.119985,0.998464,0.256306,0.637831,-0.535411,-0.956548,0.721844,-0.133906,0.718253,-0.611379,-0.579382,-0.148945,0.273827,0.229866,0.942502
2,A07_PRE,Age_22_29,22,1,0.0,0.0,44.0,43.0,46.0,50.0,49.0,42.0,40.0,84.0,-0.847244,-1.086691,-0.684206,0.121230,0.265980,-0.121754,0.162754,0.742530,1.215677,1.024537,1.027734,0.973310,0.940376,1.493668,1.291464,2.261900,0.261598,4.106214,0.263632,-0.836689,-1.461212,-0.584930,-1.256149,0.200395,1.016455,0.945267,0.781786,0.200050,0.731981,0.703101,-0.723369,0.599696,0.715970,0.715970,0.411902,-0.417636,0.533722,0.488565,0.934209,-0.080005,0.942770,0.855098,-0.487952,0.226099,-2.299490,0.714698,-0.760034,-0.274275,-1.185116,-1.954857,-0.116883,-0.305044,-0.062222,-1.550100,0.161585,-1.036417
3,A08_PRE,Age_18_21,18,1,4.0,0.0,48.0,39.0,48.0,50.0,52.0,68.0,80.0,142.0,-0.401702,0.245421,-0.684206,0.238933,0.559051,0.992762,0.162754,0.309388,0.749014,1.024537,1.027734,-0.422159,-0.183737,-1.054677,-0.647978,1.842967,1.307729,0.766394,0.918956,0.739276,0.924970,1.033904,-0.391347,2.749706,0.668256,0.656709,-0.714841,1.577136,0.484490,0.484109,-0.124601,2.329984,0.401806,0.401806,-0.350257,0.378154,-0.167554,-0.114125,-0.098478,-0.139704,0.336646,0.350923,0.684235,0.075926,-0.141362,2.995237,-1.254390,-3.487885,-2.068581,-3.926704

In [31]:
X_df = df_to_run.iloc[:, 2:14]
X_df.columns

TypeError: 'Index' object is not callable

In [7]:
all_str = str()
for idx in range(56):
    y = y_total[:, idx]

    tensorboard_path = os.path.join('tensorboard5/', str(idx) + '_' + str(int(round(time.time() % 1, 5) * 1e5)))
    print_str, _ = run_deep(X, y, idx, 0.2, 
                         train_batch_size=10, val_batch_size=5, 
                         learning_rate=1e-3, 
                         tensorboard_path=tensorboard_path, 
                         print_iteration=1, 
                         n_epochs=100,
                         tensorboard=False)
    all_str += print_str + '\n'

f = open("1e-3_fancynet3.txt", "w")
f.write(all_str)
f.close()

best of   0 => loss:        0.61, corr:  0.45, r2:  0.17
best of   1 => loss:        0.45, corr:  0.61, r2:  0.36
best of   2 => loss:        0.51, corr:  0.18, r2: -0.16
best of   3 => loss:        0.42, corr:   0.5, r2:  0.24
best of   4 => loss:      204.36, corr:   0.3, r2:  0.03
best of   5 => loss:       243.3, corr:  0.34, r2:  0.09
best of   6 => loss:      252.73, corr:  0.29, r2: -0.28
best of   7 => loss:      130.24, corr:  0.49, r2:  0.22
best of   8 => loss:       99.66, corr:  0.43, r2:  0.15
best of   9 => loss:      117.57, corr:  0.68, r2:  0.38
best of  10 => loss:       23.17, corr:  0.34, r2:   0.1
best of  11 => loss:       31.98, corr:  0.51, r2:  0.24
best of  12 => loss:      178.75, corr:  0.15, r2: -0.25
best of  13 => loss:        9.56, corr:  0.59, r2:  0.31
best of  14 => loss:       43.94, corr:  0.28, r2: -0.04
best of  15 => loss:         0.0, corr:  0.21, r2: -2.89
best of  16 => loss:         0.0, corr:  0.49, r2: -1.49
best of  17 => loss:         0.

# significance test

In [38]:
# data

df_to_run = pd.read_csv('../data/deep_test_retest_imputed.csv')
# df_to_run = pd.read_csv('data/deep_test_retest_imputed_measures_standardized.csv')
data_arr = df_to_run.iloc[:,2:].to_numpy()
X = data_arr[:, :12]
y_total = data_arr[:, 12:]

In [65]:
n_shuff = 10
results_df = dict()
results_df['name'] = ['idx', 'loss', 'p_loss', 'sig_loss', 'corr', 'p_corr', 'sig_corr', 'r2', 'p_r2', 'sig_r2']
for idx in range(5):
    y = copy.deepcopy(y_total[:, idx])

    tensorboard_path = os.path.join('tensorboard5/', str(idx) + '_' + str(int(round(time.time() % 1, 5) * 1e5)))
    _, best_r = run_deep(X, y, idx, 0.2, 
                         train_batch_size=10, val_batch_size=5, 
                         learning_rate=1e-3, 
                         tensorboard_path=tensorboard_path, 
                         print_iteration=1, 
                         n_epochs=100,
                         tensorboard=False,
                         do_print=False)
    
    all_r = dict(loss = list(), corr = list(), r2 = list())
    for i in tqdm(range(n_shuff)):
        np.random.shuffle(y)
        tensorboard_path = os.path.join('tensorboard5/', str(idx) + '_' + str(int(round(time.time() % 1, 5) * 1e5)))
        _, shuf_r = run_deep(X, y, idx, 0.2, 
                             train_batch_size=10, val_batch_size=5, 
                             learning_rate=1e-3, 
                             tensorboard_path=tensorboard_path, 
                             print_iteration=1, 
                             n_epochs=100,
                             tensorboard=False,
                             do_print=False)
        all_r['loss'].append(shuf_r['loss'])
        all_r['corr'].append(shuf_r['corr'])
        all_r['r2'].append(shuf_r['r2'])
    
    all_r['best_results'] = best_r
    col_name = df_to_run.columns[idx + 14]
    with open('results/' + col_name + '.pkl', 'wb') as f:
        pickle.dump(all_r, f)

    p_loss = 0
    p_corr = 0
    p_r2 = 0
    for i in range(n_shuff):
        p_loss += 1 if all_r['loss'][i] < best_r['loss'] else 0
        p_corr += 1 if all_r['corr'][i] > best_r['corr'] else 0
        p_r2 += 1 if all_r['r2'][i] > best_r['r2'] else 0
        
    print(p_loss, p_corr, p_r2, col_name)
    p_loss /= n_shuff
    sig_loss = 'Yes' if p_loss <= 0.05 else 'No'
    p_corr /= n_shuff
    sig_corr = 'Yes' if p_corr <= 0.05 else 'No'
    p_r2 /= n_shuff
    sig_r2 = 'Yes' if p_r2 <= 0.05 else 'No'
    
    results_df[col_name] = [int(idx), round(best_r['loss'], 4), round(p_loss, 3), sig_loss, 
                                               round(best_r['corr'], 4) , round(p_corr, 3), sig_corr,
                                               round(best_r['r2'], 4) , round(p_r2, 3), sig_r2]
    
    print(f'idx {idx} \np_value of loss: {p_loss} \np_value of corr: {p_corr} \np_value of r2: {p_r2}')
    print('\n================\n')

100%|███████████████████████████████████████████████████████████████████████████████████| 10/10 [00:07<00:00,  1.42it/s]


1 0 0 BAT_all_dprime
idx 0 
p_value of loss: 0.1 
p_value of corr: 0.0 
p_value of r2: 0.0




100%|███████████████████████████████████████████████████████████████████████████████████| 10/10 [00:07<00:00,  1.43it/s]


1 2 2 BAT_fast_dprime
idx 1 
p_value of loss: 0.1 
p_value of corr: 0.2 
p_value of r2: 0.2




100%|███████████████████████████████████████████████████████████████████████████████████| 10/10 [00:07<00:00,  1.42it/s]


6 3 4 BAT_med_dprime
idx 2 
p_value of loss: 0.6 
p_value of corr: 0.3 
p_value of r2: 0.4




100%|███████████████████████████████████████████████████████████████████████████████████| 10/10 [00:06<00:00,  1.43it/s]


6 1 1 BAT_slow_dprime
idx 3 
p_value of loss: 0.6 
p_value of corr: 0.1 
p_value of r2: 0.1




100%|███████████████████████████████████████████████████████████████████████████████████| 10/10 [00:06<00:00,  1.43it/s]

3 1 6 BAT_all_hits
idx 4 
p_value of loss: 0.3 
p_value of corr: 0.1 
p_value of r2: 0.6




# significance test v2

In [272]:
a = np.random.normal(loc=0, size=100)
b = np.random.normal(loc=0, size=100)

stats.ttest_ind(b, a, alternative='less').pvalue

0.04762672498504203

In [108]:
# data

df_to_run = pd.read_csv('../data/deep_test_retest_imputed.csv')
# df_to_run = pd.read_csv('data/deep_test_retest_imputed_measures_standardized.csv')
data_arr = df_to_run.iloc[:,2:].to_numpy()
X = data_arr[:, :12]
y_total = data_arr[:, 12:]

In [116]:
n_shuff = 10
results_df = dict()
results_df['name'] = ['idx', 
                      'loss_null', 'loss_alternative', 'p_loss', 'sig_loss', 
                      'corr_null', 'corr_alternative', 'p_corr', 'sig_corr', 
                      'r2_null', 'r2_alternative', 'p_r2', 'sig_r2']
for idx in range(5):
    y = copy.deepcopy(y_total[:, idx])
    alternative_res = dict(loss = list(), corr = list(), r2 = list())
    for i in range(n_shuff):
        tensorboard_path = os.path.join('tensorboard5/', str(idx) + '_' + str(int(round(time.time() % 1, 5) * 1e5)))
        _, res = run_deep(X, y, idx, 0.2, 
                             train_batch_size=10, val_batch_size=5, 
                             learning_rate=1e-3, 
                             tensorboard_path=tensorboard_path, 
                             print_iteration=1, 
                             n_epochs=100,
                             tensorboard=False,
                             do_print=False)
        alternative_res['loss'].append(res['loss'])
        alternative_res['corr'].append(res['corr'])
        alternative_res['r2'].append(res['r2'])
    
    null_res = dict(loss = list(), corr = list(), r2 = list())
    for i in tqdm(range(n_shuff)):
        np.random.shuffle(y)
        tensorboard_path = os.path.join('tensorboard5/', str(idx) + '_' + str(int(round(time.time() % 1, 5) * 1e5)))
        _, shuf_r = run_deep(X, y, idx, 0.2, 
                             train_batch_size=10, val_batch_size=5, 
                             learning_rate=1e-3, 
                             tensorboard_path=tensorboard_path, 
                             print_iteration=1, 
                             n_epochs=100,
                             tensorboard=False,
                             do_print=False)
        null_res['loss'].append(shuf_r['loss'])
        null_res['corr'].append(shuf_r['corr'])
        null_res['r2'].append(shuf_r['r2'])
    
    
    all_res = dict(alternative=alternative_res, null=null_res)
    
    col_name = df_to_run.columns[idx + 14]
    with open('results/' + col_name + '.pkl', 'wb') as f:
        pickle.dump(all_res, f)

    p_loss = stats.ttest_ind(alternative_res['loss'], null_res['loss'], alternative='less').pvalue
    p_corr = stats.ttest_ind(alternative_res['corr'], null_res['corr'], alternative='greater').pvalue
    p_r2 = stats.ttest_ind(alternative_res['r2'], null_res['r2'], alternative='greater').pvalue

    sig_loss = 'Yes' if p_loss <= 0.05 else 'No'
    sig_corr = 'Yes' if p_corr <= 0.05 else 'No'
    sig_r2 = 'Yes' if p_r2 <= 0.05 else 'No'
    
    
    results_df[col_name] = [int(idx), 
                            round(np.mean(null_res['loss']), 4), round(np.mean(alternative_res['loss']), 4), round(p_loss, 3), sig_loss, 
                            round(np.mean(null_res['corr']), 4), round(np.mean(alternative_res['corr']), 4), round(p_corr, 3), sig_corr,
                            round(np.mean(null_res['r2']), 4), round(np.mean(alternative_res['r2']), 4) , round(p_r2, 3), sig_r2]
    
    print(f'idx {idx} \np_value of loss: {p_loss} \np_value of corr: {p_corr} \np_value of r2: {p_r2}')
    print('\n================\n')

100%|███████████████████████████████████████████████████████████████████████████████████| 10/10 [00:06<00:00,  1.44it/s]


idx 0 
p_value of loss: 0.0758283425650643 
p_value of corr: 0.0030275591244975993 
p_value of r2: 0.00012036604986994324




100%|███████████████████████████████████████████████████████████████████████████████████| 10/10 [00:06<00:00,  1.44it/s]


idx 1 
p_value of loss: 0.009732692797546046 
p_value of corr: 0.0002578654558731825 
p_value of r2: 0.0011933489142811444




100%|███████████████████████████████████████████████████████████████████████████████████| 10/10 [00:06<00:00,  1.44it/s]


idx 2 
p_value of loss: 0.8667914709297857 
p_value of corr: 0.004904476514091272 
p_value of r2: 0.03894724407054126




100%|███████████████████████████████████████████████████████████████████████████████████| 10/10 [00:07<00:00,  1.43it/s]


idx 3 
p_value of loss: 0.0184246701971701 
p_value of corr: 0.00046407838815542846 
p_value of r2: 0.0011241638437281287




100%|███████████████████████████████████████████████████████████████████████████████████| 10/10 [00:06<00:00,  1.44it/s]

idx 4 
p_value of loss: 0.5957577475191669 
p_value of corr: 0.0001770622899225477 
p_value of r2: 0.5338939196270962




In [117]:
pd.DataFrame(results_df)

,name,BAT_all_dprime,BAT_fast_dprime,BAT_med_dprime,BAT_slow_dprime,BAT_all_hits
0,idx,0,1,2,3,4
1,loss_null,0.7915,0.629,0.5352,0.5436,261.1651
2,loss_alternative,0.6552,0.4659,0.6131,0.4126,284.3998
3,p_loss,0.076,0.01,0.867,0.018,0.596
4,sig_loss,No,Yes,No,Yes,No
5,corr_null,0.3925,0.184,0.1253,0.3067,0.1678
6,corr_alternative,0.5768,0.4924,0.3495,0.5487,0.4483
7,p_corr,0.003,0.0,0.005,0.0,0.0
8,sig_corr,Yes,Yes,Yes,Yes,Yes
9,r2_null,0.0093,-0.055,-0.1276,0.0303,-0.2174


In [115]:
all_res

{'alternative': {'loss': [144.6947021484375,
   92.77509657541911,
   210.32421239217123,
   149.25953102111816,
   100.41545232137044,
   188.40666516621908,
   210.6069793701172,
   123.49387486775716,
   158.8606859842936,
   173.4744784037272],
  'corr': [0.5011272500073093,
   0.5321664680610092,
   0.5007289658751364,
   0.40957389927775456,
   0.5955573548215631,
   0.3910435094215344,
   0.4299930070391786,
   0.5833155251476821,
   0.16974276822598033,
   0.4413905224729217],
  'r2': [0.1955723200149091,
   0.25362383264387156,
   0.2388478200253179,
   0.041781467801786154,
   0.3543207407113136,
   0.09394382319709038,
   0.16687318973617393,
   0.27649113659563296,
   -0.8911432920242215,
   0.06869470113911469]},
 'null': {'loss': [303.03292083740234,
   442.87876383463544,
   187.95297876993814,
   234.6160888671875,
   233.20376904805502,
   230.94719314575195,
   225.9209950764974,
   260.98717244466144,
   267.67936579386395,
   255.38497670491537],
  'corr': [-0.04998